In [1]:
# import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import re
import string

print(tf.__version__)

2.15.0


In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2024-01-06 06:42:35--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv’

train-data.tsv      100%[===================>] 349.84K  --.-KB/s    in 0.06s   

2024-01-06 06:42:36 (5.89 MB/s) - ‘train-data.tsv’ saved [358233/358233]

--2024-01-06 06:42:36--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv’

valid-data.tsv      100%[==============

In [3]:
# load data into pandas
train = pd.read_csv(
    train_file_path,
    encoding = "ISO-8859-1",
    sep="\t",
    header=0,
    names=['type', 'message'],
    usecols=['type', 'message'],
    dtype={'type': 'str', 'message': 'str'}
    )

test = pd.read_csv(
    test_file_path,
    encoding = "ISO-8859-1",
    sep="\t",
    header=0,
    names=['type', 'message'],
    usecols=['type', 'message'],
    dtype={'type': 'str', 'message': 'str'}
    )

In [4]:
label_map = {
    'ham' : 0,
    'spam' : 1
}

train['type'] = train['type'].map(label_map)
test['type'] = test['type'].map(label_map)

In [5]:
import re
import string
# Create a custom standardization function to strip HTML break tags ''.
def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  stripped_html = tf.strings.regex_replace(lowercase, '', ' ')
  return tf.strings.regex_replace(stripped_html,
                                  '[%s]' % re.escape(string.punctuation), '')

# Vocabulary size and number of words in a sequence.
vocab_size = 10000
sequence_length = 150

# Use the text vectorization layer to normalize, split, and map strings to
# integers. Note that the layer uses the custom standardization defined above.
# Set maximum_sequence length as all samples are not of the same length.
vectorize_layer = keras.layers.experimental.preprocessing.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
train_array = np.array(train.message)
vectorize_layer.adapt(train_array)

In [6]:
# build our model
model = keras.Sequential([
                          vectorize_layer,
                          keras.layers.Embedding(
                              # Vocabulary size
                              vocab_size,
                              # dimensions
                              32,
                              name="embedding"
                          ),
                          keras.layers.LSTM(32),
                          keras.layers.Dense(1, activation="sigmoid")
                          ])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization (TextVe  (None, 150)               0         
 ctorization)                                                    
                                                                 
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 lstm (LSTM)                 (None, 32)                8320      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 328353 (1.25 MB)
Trainable params: 328353 (1.25 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [21]:
# train the model
model.compile(loss="binary_crossentropy",
              optimizer="rmsprop",
              metrics=['accuracy'])

monitor = keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=1e-4, patience=25, verbose=1, mode='max', restore_best_weights=True)

history = model.fit(train["message"], train["type"], callbacks=[monitor], epochs=100, validation_split=0.2)

Epoch 1/100
105/105 [==============================] - 11s 88ms/step - loss: 0.1002 - accuracy: 0.9755 - val_loss: 0.1227 - val_accuracy: 0.9689
Epoch 2/100
105/105 [==============================] - 7s 70ms/step - loss: 0.0994 - accuracy: 0.9722 - val_loss: 0.0683 - val_accuracy: 0.9761
Epoch 3/100
105/105 [==============================] - 8s 80ms/step - loss: 0.0868 - accuracy: 0.9740 - val_loss: 0.0803 - val_accuracy: 0.9737
Epoch 4/100
105/105 [==============================] - 7s 66ms/step - loss: 0.0878 - accuracy: 0.9761 - val_loss: 0.0736 - val_accuracy: 0.9833
Epoch 5/100
105/105 [==============================] - 8s 78ms/step - loss: 0.0918 - accuracy: 0.9773 - val_loss: 0.0890 - val_accuracy: 0.9785
Epoch 6/100
105/105 [==============================] - 9s 85ms/step - loss: 0.0904 - accuracy: 0.9758 - val_loss: 0.0703 - val_accuracy: 0.9856
Epoch 7/100
105/105 [==============================] - 7s 67ms/step - loss: 0.0808 - accuracy: 0.9776 - val_loss: 0.0650 - val_accuracy

In [22]:
loss, accuracy = model.evaluate(test.message, test['type'])

print("Loss: ", loss)
print("Accuracy: ", accuracy)

44/44 [==============================] - 1s 16ms/step - loss: 0.0685 - accuracy: 0.9777
Loss:  0.068510040640831
Accuracy:  0.9777138829231262


In [23]:
model.predict(train_array[:10])

1/1 [==============================] - 0s 461ms/step


array([[0.00567949],
       [0.00637559],
       [0.00900586],
       [0.00612427],
       [0.03738533],
       [0.02093217],
       [0.0058374 ],
       [0.96428156],
       [0.00834286],
       [0.00650666]], dtype=float32)

In [39]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  pred = model.predict([pred_text])[0]

  return [pred[0], 'spam' if round(pred[0], 3) >= 0.007 else 'ham']

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

1/1 [==============================] - 0s 41ms/step
[0.005619218, 'ham']


In [40]:
test_messages = ["how are you doing today",
                 "sale today! to stop texts call 98912460324",
                 "i dont want to go. can we try it a different day? available sat",
                 "our new mobile video service is live. just install on your phone to start watching.",
                 "you have won £1000 cash! call to claim your prize.",
                 "i'll bring it tomorrow. don't forget the milk.",
                 "wow, is your arm alright. that happened to me one time too"
                ]

for i in range(len(test_messages)):
  print(predict_message(test_messages[i]))

1/1 [==============================] - 0s 41ms/step
[0.005619218, 'ham']
1/1 [==============================] - 0s 40ms/step
[0.58190835, 'spam']
1/1 [==============================] - 0s 42ms/step
[0.0061330595, 'ham']
1/1 [==============================] - 0s 39ms/step
[0.007174769, 'spam']
1/1 [==============================] - 0s 39ms/step
[0.091603644, 'spam']
1/1 [==============================] - 0s 45ms/step
[0.006156772, 'ham']
1/1 [==============================] - 0s 55ms/step
[0.005924247, 'ham']


In [41]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


1/1 [==============================] - 0s 79ms/step
You passed the challenge. Great job!
